# Memory Management in LangGraph

## Notebook Roadmap

1. Memory Foundations
2. Plain Python Memory Demonstration
3. LangGraph Memory Semantics
4. Persistent Long-Term Memory with SQLite
5. Persistence Versus Memory Scope
6. Custom Long-Term Memory with LangGraph Store
7. Manual State-Based Memory in LangGraph
8. Pure LangGraph Custom Memory Manager
9. LangGraph Versus LangMem

> All original notebook cells and their contents are preserved below. The section cells only organize the learning flow.

---
## 1. Memory Foundations

Short-term memory, long-term memory, comparison, and a simple human analogy.

Short-term memory = temporarily useful context for the current interaction/task/session.

Examples:
- recent chat messages
- current task state
- current tool outputs
- temporary variables
- current document context

Simple line:

Short-term memory stores information needed for the current interaction or task.

Example:

User: My name is Sunny.
AI: Hi Sunny.

User: What is my name?
AI: Your name is Sunny.

Here, the context of the current conversation is short-term memory.

Short-term memory = what the AI needs to remember in the current conversation.

Example:

You: My name is Sunny.
AI: Hi Sunny.

You: What is my name?
AI: Sunny.

Here, "My name is Sunny" is remembered within the context of the current chat.

Long-term memory = what the AI can remember in the future, even in a new chat/session.

Example:

Today:
You: I prefer Python.

Next week, new chat:
You: Which language should I use?

AI:
You prefer Python, so Python may suit you.

Long-term memory means saving important information from previous conversations so that it can be reused in future conversations.

Conversation
   ↓
Important facts / preferences / decisions
   ↓
Long-term memory
   ↓
Used again in future chats

Short-term = memory of the current conversation

Long-term = reusable memory for future conversations

Long-term memory = information that remains useful in future interactions/tasks and can be retrieved/reused later.

Examples:
- user preferences
- personal facts
- previous decisions
- learned behavior
- past experiences
- project information
- recurring requirements

Simple line:

Long-term memory stores useful information that can be retained and reused beyond the current interaction or session.

Example:

Today:
User: I prefer Python.

3 days later:
User: Which language should I use?

AI can retrieve:
User prefers Python.

Short-term:
Someone just told you a phone number,
and you remembered it for 2 minutes.

Long-term:
You remember your home address or birthday
for years.

| Short-term memory       | Long-term memory               |
| ----------------------- | ------------------------------ |
| Current context         | Reusable future context        |
| Recent interactions     | Important retained information |
| Usually temporary       | Usually persistent/durable     |
| Limited working context | Can grow much larger           |
| Current task/session    | Across tasks/sessions          |


Short-term memory helps the AI remember what is happening now, while long-term memory helps it remember useful information from the past for future interactions.”

And a human analogy:

Short-term memory
→ What are we talking about right now?

Long-term memory
→ What do I already know or remember from before?

---
## 2. Plain Python Memory Demonstration

A framework-independent example of thread-scoped and user-scoped memory.

In [ ]:
# ============================================================
# PURE PYTHON MEMORY MANAGEMENT
# NO LANGCHAIN
# NO LANGGRAPH
# ============================================================


# ============================================================
# SHORT-TERM MEMORY
# thread_id based
# ============================================================

class ShortTermMemory:

    def __init__(self):
        self.threads = {}


    def add_message(
        self,
        thread_id,
        role,
        content
    ):

        if thread_id not in self.threads:
            self.threads[thread_id] = []

        self.threads[thread_id].append(
            {
                "role": role,
                "content": content
            }
        )


    def get_messages(
        self,
        thread_id
    ):

        return self.threads.get(
            thread_id,
            []
        )


# ============================================================
# LONG-TERM MEMORY
# user_id based
# ============================================================

class LongTermMemory:

    def __init__(self):
        self.users = {}


    def add_memory(
        self,
        user_id,
        memory
    ):

        if user_id not in self.users:
            self.users[user_id] = []

        if memory not in self.users[user_id]:

            self.users[user_id].append(
                memory
            )


    def get_memories(
        self,
        user_id
    ):

        return self.users.get(
            user_id,
            []
        )


# ============================================================
# CREATE MEMORY OBJECTS
# ============================================================

short_term_memory = ShortTermMemory()

long_term_memory = LongTermMemory()


# ============================================================
# USER
# ============================================================

user_id = "sunny_123"


# ============================================================
# THREAD 1
# ============================================================

thread_1 = "thread_1"


short_term_memory.add_message(
    thread_1,
    "user",
    "My name is Sunny."
)


short_term_memory.add_message(
    thread_1,
    "assistant",
    "Nice to meet you Sunny."
)


# Save useful information
# as long-term user memory

long_term_memory.add_memory(
    user_id,
    "User's name is Sunny."
)


long_term_memory.add_memory(
    user_id,
    "User prefers Python."
)


# ============================================================
# THREAD 2
# NEW CONVERSATION
# ============================================================

thread_2 = "thread_2"


short_term_memory.add_message(
    thread_2,
    "user",
    "Let's discuss AWS."
)


short_term_memory.add_message(
    thread_2,
    "assistant",
    "Sure, let's discuss AWS."
)


# ============================================================
# CHECK SHORT-TERM MEMORY
# ============================================================

print(
    "\nTHREAD 1 MEMORY:"
)

for message in short_term_memory.get_messages(
    thread_1
):
    print(message)


print(
    "\nTHREAD 2 MEMORY:"
)

for message in short_term_memory.get_messages(
    thread_2
):
    print(message)


# ============================================================
# CHECK LONG-TERM MEMORY
# ============================================================

print(
    "\nLONG-TERM MEMORY FOR SUNNY:"
)

for memory in long_term_memory.get_memories(
    user_id
):
    print(memory)

Output conceptually:

THREAD 1 MEMORY:

{'role': 'user', 'content': 'My name is Sunny.'}
{'role': 'assistant', 'content': 'Nice to meet you Sunny.'}


THREAD 2 MEMORY:

{'role': 'user', 'content': "Let's discuss AWS."}
{'role': 'assistant', 'content': "Sure, let's discuss AWS."}


LONG-TERM MEMORY FOR SUNNY:

User's name is Sunny.
User prefers Python.

user_id = sunny_123

             Sunny
               |
       -----------------
       |               |
   thread_1         thread_2
       |               |
       ↓               ↓
"My name..."       "Discuss AWS"
       |               |
 Short-Term        Short-Term
   Memory            Memory

Short-term memory of thread_1:

My name is Sunny.
Nice to meet you Sunny.

Short-term memory of thread_2:

Let's discuss AWS.
Sure...

Thread 2 does not automatically see the conversation history of Thread 1.

However, long-term memory:

user_id = sunny_123

- User's name is Sunny.
- User prefers Python.

belongs to the user, not to any single thread.

Therefore, even a new thread_100 can use:

long_term_memory.get_memories(
    "sunny_123"
)

to access the same memories.

This proves the exact concept:
SHORT-TERM MEMORY

thread_id
   ↓
Specific conversation

versus:

LONG-TERM MEMORY

user_id
   ↓
Information reusable across conversations

Another important point: both are currently Python dictionaries, so both will be lost on restart.

Short-term scope ≠ storage duration
Long-term scope  ≠ automatically persistent

If we replace LongTermMemory with SQLite/PostgreSQL in this pure Python example, then:

user_id
   ↓
SQLite/PostgreSQL
   ↓
User memory
   ↓
available even after restart

You can say this line in class:

“Short-term and long-term memory are not LangGraph concepts only. Even in a plain Python application, we can scope conversation memory by thread_id and reusable user memory by user_id. LangGraph simply provides built-in abstractions to manage the same pattern more conveniently.”

And the shortest version:

thread_id answers: which conversation?
user_id answers: which user?

---
## 3. LangGraph Memory Semantics

Checkpoint, Checkpointer, Store, `thread_id`, `user_id`, scope, and durability.

In LangGraph terminology, short-term versus long-term memory is primarily about recall scope, not simply how long the data remains stored. A checkpointer persists state within the same thread, while a Store is designed for memories that can be reused across different threads or sessions.

Short-term memory = thread-scoped memory
Long-term memory = information shared across threads/sessions

According to the LangGraph documentation, short-term memory is part of the agent state and is persisted through a checkpointer so that the same thread_id can be resumed later. Production examples even use PostgresSaver, so the data can remain durable after a restart — yet the documentation still calls it short-term/thread-level memory.

And the official long-term memory definition:

user-specific or application-level information that can be accessed across different threads and sessions.

LangGraph uses a Store for this purpose.

The official Functional API documentation is even more direct:

Short-term memory allows storing information across different invocations of the same thread id.
Long-term memory allows storing information across different thread ids.

Sunny
│
├── thread_1
│    "My favorite language is Python"
│
├── thread_2
│    "Let's discuss AWS"
│
└── thread_3
     "Build a RAG system"

Use PostgresSaver to persist thread_1 for 2 years:

thread_1
→ persisted in PostgreSQL
→ restart safe

In LangGraph terminology, it is still:

Short-term / thread-scoped memory

because it belongs to that specific thread.

In the long-term Store:

user_id = sunny_123

- prefers Python
- working on Agentic RAG
- prefers concise answers

this information:

thread_1
thread_2
thread_3

can be retrieved in all of them. That is why LangGraph calls it cross-thread long-term memory.

The production perspective has also been verified: the official documentation shows a database-backed checkpointer such as PostgresSaver for short-term state in production, and a persistent Store such as PostgresStore for long-term memory.

What exactly is a checkpoint?

In LangGraph, a checkpoint = a saved snapshot of the graph state.

Suppose graph state:

state = {
    "messages": [...],
    "current_step": "approval",
    "document_id": "DOC-101"
}

The checkpointer saves this state against a thread_id:

thread_id = chat_001

Checkpoint
    ↓
messages
current_step
document_id
other graph state

Its main purpose is:

“Continue this particular conversation/workflow later from the same state.”

Example:

Thread-1

User: My name is Sunny.
AI: Hi Sunny.

User: I am learning LangGraph.
AI: Great.

            ↓

        Checkpointer

            ↓

thread_1 state saved

Now, in the same thread:

thread_id = thread_1

User:
What am I learning?

The checkpointer will load the previous state:

My name is Sunny.
I am learning LangGraph.

AI:

You are learning LangGraph.

This is short-term/conversation memory.

But if I use PostgresSaver, will it remain available after a restart?

Absolutely.

InMemorySaver
→ lost on restart

PostgresSaver
→ remains saved after restart

However, even after using PostgresSaver, we will usually still call it short-term/thread memory.

Why?

Because its scope is still:

thread_1

rather than:

all conversations of Sunny

This is the key point.

Consider this example:

User:

user_id = sunny_123

The user has 3 chats:

Sunny
 │
 ├── thread_1
 │     My favorite language is Python.
 │
 ├── thread_2
 │     Discuss AWS.
 │
 └── thread_3
       Discuss RAG.

Checkpointer:

thread_1
→ knows thread_1 history

thread_2
→ knows thread_2 history

thread_3
→ knows thread_3 history

Now Sunny asks in thread_2:

Which programming language do I prefer?

The checkpointer for thread_2 will not automatically see the history of thread_1.

Even if thread_1 is permanently stored in PostgreSQL.

That's why:

Persistent ≠ Long-term memory

What is long-term memory?

Long-term memory is stored for user-level or cross-thread reuse.

Example:

user_id = sunny_123

Long-Term Store:

- User prefers Python.
- User is building an Agentic RAG project.
- User prefers concise explanations.

Now:

Thread-1
Thread-2
Thread-3
Thread-100

Regardless of the thread, relevant memories for the same user_id can be retrieved.

                 user_id = sunny_123
                         │
                         ▼
                    Long-Term Store
                         │
          "User prefers Python"
                         │
            ┌────────────┼────────────┐
            ▼            ▼            ▼
        Thread-1     Thread-2     Thread-3

|                                          | Checkpointer             | Long-Term Store                     |
| ---------------------------------------- | ------------------------ | ----------------------------------- |
| Key concept                              | Conversation state       | User/app memory                     |
| Scope                                    | **One thread**           | **Across threads**                  |
| Identifier                               | `thread_id`              | Usually `user_id` / namespace       |
| Stores                                   | Messages + graph state   | Selected facts/preferences/memories |
| Same thread recall                       | ✅                        | ✅ if retrieved                      |
| New thread automatically gets old thread | ❌                        | ✅ can retrieve same user's memory   |
| Can be persistent?                       | ✅                        | ✅                                   |
| Memory category                          | Short-term/thread memory | Long-term/cross-thread memory       |


Why call it “short-term” if it can stay for years?

Because “short-term” here basically means:

conversation-scoped memory

not necessarily:

“data will disappear after 10 minutes.”

Suppose:

thread_1 saved for 2 years

It is technically durable.

But it still represents:

Conversation #1

not a reusable model of:

Sunny as a user

Therefore, in memory taxonomy, it is still thread-level/short-term memory.

Real-life analogy

Checkpointer = one notebook for one meeting

Meeting-1 notebook
Meeting-2 notebook
Meeting-3 notebook

You can keep every notebook for 10 years.

Still each notebook is about that meeting.

Long-term Store = employee/user profile

Sunny:
- prefers Python
- AI Engineer
- building RAG

This can be reused in every meeting.

                         USER
                    user_id = Sunny
                          │
          ┌───────────────┴───────────────┐
          │                               │
          ▼                               ▼

      Thread-1                        Thread-2
          │                               │
          ▼                               ▼
     Checkpointer                    Checkpointer
          │                               │
    conversation-1                  conversation-2
      state/history                   state/history

          └───────────────┬───────────────┘
                          │
                          ▼

                     Long-Term Store
                          │
                   user_id = Sunny
                          │
                  reusable memories

A checkpointer can persist data for a very long time, but it is still considered thread-level memory because its state belongs to one specific conversation. Long-term memory is different because it stores reusable information that can be accessed across multiple conversations or threads.

---
## 4. Persistent Long-Term Memory with SQLite

A complete custom SQLite-backed implementation with extraction and semantic retrieval.

Yes. SQLite is the easiest setup because it does not require a separate database server, Docker, or a username/password. Python already includes sqlite3.

We will keep your existing architecture exactly the same:

SHORT-TERM
memory + summary + full_history
→ current running conversation

LONG-TERM
SQLite database
→ persistent across restart
→ user_id based
→ semantic retrieval

Important: SQLite is excellent for demonstrating persistence, but for high-scale production, the LangGraph documentation recommends a persistent database-backed store such as AsyncPostgresStore.

In [ ]:
# ============================================================
# INSTALL
# ============================================================
# pip install -U langgraph langchain-openai pydantic


# ============================================================
# IMPORTS
# ============================================================

import sqlite3
import uuid
import json
import math

from dataclasses import dataclass
from typing import TypedDict, List, Dict

from pydantic import BaseModel, Field

from langchain_openai import (
    ChatOpenAI,
    OpenAIEmbeddings
)

from langchain_core.messages import SystemMessage

from langgraph.graph import (
    StateGraph,
    START,
    END
)

from langgraph.runtime import Runtime


# ============================================================
# MODEL
# ============================================================

model = ChatOpenAI(
    model="gpt-4.1-mini",
    temperature=0
)


# ============================================================
# EMBEDDING MODEL
# ============================================================

embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small"
)


# ============================================================
# CONFIG
# ============================================================

MAX_RECENT_MESSAGES = 6


# ============================================================
# USER CONTEXT
# ============================================================

@dataclass
class UserContext:

    user_id: str


# ============================================================
# LANGGRAPH STATE
# ============================================================
#
# This is SHORT-TERM conversation memory.
# ============================================================

class MemoryState(TypedDict):

    user_input: str

    # Latest raw messages
    memory: List[Dict]

    # Summary of old messages
    summary: str

    # Complete current conversation
    full_history: List[Dict]

    answer: str


# ============================================================
# LONG-TERM MEMORY DATABASE
# ============================================================
#
# SQLite is persistent.
#
# The memory will remain in the database
# even after the application restarts.
# ============================================================

class SQLiteLongTermMemory:

    def __init__(
        self,
        db_path="long_term_memory.db"
    ):

        self.db_path = db_path

        self.create_table()


    # --------------------------------------------------------
    # DATABASE CONNECTION
    # --------------------------------------------------------

    def connect(self):

        return sqlite3.connect(
            self.db_path
        )


    # --------------------------------------------------------
    # CREATE TABLE
    # --------------------------------------------------------

    def create_table(self):

        with self.connect() as connection:

            connection.execute(
                """
                CREATE TABLE IF NOT EXISTS memories (

                    memory_id TEXT PRIMARY KEY,

                    user_id TEXT NOT NULL,

                    content TEXT NOT NULL,

                    embedding TEXT NOT NULL,

                    created_at TIMESTAMP
                    DEFAULT CURRENT_TIMESTAMP

                )
                """
            )


            connection.commit()


    # --------------------------------------------------------
    # SAVE MEMORY
    # --------------------------------------------------------

    def add_memory(
        self,
        user_id,
        content
    ):

        # ------------------------------------
        # Check duplicate
        # ------------------------------------

        with self.connect() as connection:

            existing = connection.execute(
                """
                SELECT memory_id

                FROM memories

                WHERE user_id = ?
                AND content = ?
                """,

                (
                    user_id,
                    content
                )

            ).fetchone()


        if existing:

            return


        # ------------------------------------
        # Create embedding
        # ------------------------------------

        vector = embeddings.embed_query(
            content
        )


        # Convert vector to JSON string

        vector_json = json.dumps(
            vector
        )


        # ------------------------------------
        # Save in SQLite
        # ------------------------------------

        memory_id = str(
            uuid.uuid4()
        )


        with self.connect() as connection:

            connection.execute(
                """
                INSERT INTO memories (

                    memory_id,
                    user_id,
                    content,
                    embedding

                )

                VALUES (?, ?, ?, ?)
                """,

                (
                    memory_id,
                    user_id,
                    content,
                    vector_json
                )
            )


            connection.commit()


    # --------------------------------------------------------
    # COSINE SIMILARITY
    # --------------------------------------------------------

    def cosine_similarity(
        self,
        vector_a,
        vector_b
    ):

        dot_product = sum(

            a * b

            for a, b
            in zip(
                vector_a,
                vector_b
            )
        )


        norm_a = math.sqrt(
            sum(
                a * a
                for a in vector_a
            )
        )


        norm_b = math.sqrt(
            sum(
                b * b
                for b in vector_b
            )
        )


        if norm_a == 0 or norm_b == 0:

            return 0


        return (
            dot_product
            /
            (norm_a * norm_b)
        )


    # --------------------------------------------------------
    # SEMANTIC MEMORY SEARCH
    # --------------------------------------------------------

    def search(
        self,
        user_id,
        query,
        limit=5
    ):

        # Query embedding

        query_vector = embeddings.embed_query(
            query
        )


        # Get all memories for this user

        with self.connect() as connection:

            rows = connection.execute(
                """
                SELECT
                    content,
                    embedding

                FROM memories

                WHERE user_id = ?
                """,

                (
                    user_id,
                )

            ).fetchall()


        scored_memories = []


        for content, embedding_json in rows:

            memory_vector = json.loads(
                embedding_json
            )


            score = self.cosine_similarity(

                query_vector,

                memory_vector
            )


            scored_memories.append(
                (
                    score,
                    content
                )
            )


        # Highest similarity first

        scored_memories.sort(
            key=lambda item: item[0],
            reverse=True
        )


        return [

            content

            for score, content
            in scored_memories[:limit]
        ]


    # --------------------------------------------------------
    # GET ALL MEMORIES
    # --------------------------------------------------------

    def get_all(
        self,
        user_id
    ):

        with self.connect() as connection:

            rows = connection.execute(
                """
                SELECT content

                FROM memories

                WHERE user_id = ?

                ORDER BY created_at
                """,

                (
                    user_id,
                )

            ).fetchall()


        return [

            row[0]

            for row in rows
        ]


# ============================================================
# CREATE DATABASE OBJECT
# ============================================================

long_term_memory = SQLiteLongTermMemory(
    "long_term_memory.db"
)


# ============================================================
# LONG-TERM MEMORY EXTRACTION SCHEMA
# ============================================================

class ExtractedMemories(BaseModel):

    memories: List[str] = Field(
        default_factory=list,
        description="""
        Important long-term information
        learned about the user.
        """
    )


memory_extractor = (
    model.with_structured_output(
        ExtractedMemories
    )
)


# ============================================================
# SHORT-TERM SUMMARY
# ============================================================

def create_summary(
    old_messages,
    existing_summary
):

    conversation_text = "\n".join(

        f"{msg['role']}: {msg['content']}"

        for msg in old_messages
    )


    prompt = f"""
You maintain short-term conversation memory.

Existing summary:

{existing_summary or "No previous summary."}


Older conversation:

{conversation_text}


Create one concise updated summary.

Preserve:

- important conversation context
- facts
- decisions
- projects
- preferences

Ignore greetings and unnecessary small talk.
"""


    response = model.invoke(
        prompt
    )


    return response.content.strip()


# ============================================================
# CUSTOM LONG-TERM MEMORY EXTRACTION
# ============================================================

def extract_long_term_memories(
    user_message
):

    prompt = f"""
Analyze this user message:

{user_message}


Extract only useful LONG-TERM information
about the user.

Store things such as:

- preferences
- personal facts
- goals
- ongoing projects
- skills
- recurring requirements
- important decisions

Do NOT store:

- greetings
- general knowledge questions
- temporary questions
- one-time requests
- unnecessary small talk


Example:

User:
I prefer Python for AI development.

Memory:
User prefers Python for AI development.


User:
What is LangGraph?

Memory:
Nothing.


If there is nothing useful,
return an empty list.
"""


    result = memory_extractor.invoke(
        prompt
    )


    return result.memories


# ============================================================
# NODE 1
# CHATBOT
# ============================================================

def chatbot(
    state: MemoryState,
    runtime: Runtime[UserContext]
):

    user_id = runtime.context.user_id


    # ========================================================
    # SHORT-TERM MEMORY
    # ========================================================

    memory = list(
        state["memory"]
    )


    full_history = list(
        state["full_history"]
    )


    # Add current user message

    user_message = {

        "role": "user",

        "content": state["user_input"]
    }


    memory.append(
        user_message
    )


    full_history.append(
        user_message
    )


    # ========================================================
    # LONG-TERM MEMORY RETRIEVAL
    # ========================================================

    long_term_memories = (
        long_term_memory.search(

            user_id=user_id,

            query=state["user_input"],

            limit=5
        )
    )


    if long_term_memories:

        long_term_text = "\n".join(

            f"- {item}"

            for item
            in long_term_memories
        )

    else:

        long_term_text = (
            "No relevant long-term memories."
        )


    # ========================================================
    # BUILD LLM CONTEXT
    # ========================================================

    system_message = SystemMessage(
        content=f"""
You are a helpful AI assistant.


SHORT-TERM CONVERSATION SUMMARY:

{state["summary"] or "No previous summary."}


RELEVANT LONG-TERM USER MEMORIES:

{long_term_text}


Use long-term memories only when relevant.

Use recent messages to maintain
conversation continuity.
"""
    )


    response = model.invoke(

        [
            system_message,
            *memory
        ]
    )


    # ========================================================
    # SAVE ASSISTANT MESSAGE
    # ========================================================

    assistant_message = {

        "role": "assistant",

        "content": response.content
    }


    memory.append(
        assistant_message
    )


    full_history.append(
        assistant_message
    )


    return {

        "memory": memory,

        "full_history": full_history,

        "answer": response.content
    }


# ============================================================
# NODE 2
# SHORT-TERM MEMORY COMPRESSION
# ============================================================

def compress_memory(
    state: MemoryState
):

    memory = list(
        state["memory"]
    )


    if len(memory) <= MAX_RECENT_MESSAGES:

        return {}


    # Everything except latest 6

    old_messages = memory[
        :-MAX_RECENT_MESSAGES
    ]


    # Latest 6 raw messages

    recent_messages = memory[
        -MAX_RECENT_MESSAGES:
    ]


    # Update summary

    updated_summary = create_summary(

        old_messages,

        state["summary"]
    )


    return {

        "memory": recent_messages,

        "summary": updated_summary
    }


# ============================================================
# NODE 3
# LONG-TERM MEMORY UPDATE
# ============================================================

def update_long_term_memory(
    state: MemoryState,
    runtime: Runtime[UserContext]
):

    user_id = runtime.context.user_id


    # Extract important facts

    memories = extract_long_term_memories(
        state["user_input"]
    )


    # Save in SQLite

    for memory in memories:

        long_term_memory.add_memory(

            user_id=user_id,

            content=memory
        )


    return {}


# ============================================================
# BUILD LANGGRAPH
# ============================================================

builder = StateGraph(

    MemoryState,

    context_schema=UserContext
)


builder.add_node(
    "chatbot",
    chatbot
)


builder.add_node(
    "compress_memory",
    compress_memory
)


builder.add_node(
    "update_long_term_memory",
    update_long_term_memory
)


# ============================================================
# GRAPH FLOW
# ============================================================

builder.add_edge(
    START,
    "chatbot"
)


builder.add_edge(
    "chatbot",
    "compress_memory"
)


builder.add_edge(
    "compress_memory",
    "update_long_term_memory"
)


builder.add_edge(
    "update_long_term_memory",
    END
)


# ============================================================
# COMPILE
# ============================================================

graph = builder.compile()


# ============================================================
# INITIAL SHORT-TERM STATE
# ============================================================

state = {

    "user_input": "",

    "memory": [],

    "summary": "",

    "full_history": [],

    "answer": ""
}


# ============================================================
# USER ID
# ============================================================

context = UserContext(
    user_id="sunny_123"
)


# ============================================================
# CHAT LOOP
# ============================================================

while True:

    user_input = input(
        "\nYou: "
    ).strip()


    if user_input.lower() in {
        "exit",
        "quit"
    }:

        print(
            "Chat ended."
        )

        break


    if not user_input:

        continue


    state["user_input"] = (
        user_input
    )


    state = graph.invoke(

        state,

        context=context
    )


    print(
        "\nAI:",
        state["answer"]
    )


    print(
        "\nRecent Messages:",
        len(state["memory"])
    )


    print(
        "Full History:",
        len(state["full_history"])
    )


    if state["summary"]:

        print(
            "\nCurrent Summary:\n",
            state["summary"]
        )

The test is now very simple.

Run it for the first time:

You:
My favorite programming language is Python.

You:
I am building an Agentic RAG project.

Close the program:

exit

The following file:

long_term_memory.db

will be created automatically in your folder.

Now restart the Python program.

Short-term:

memory = []
summary = ""
full_history = []

will be reset.

But SQLite:

long_term_memory.db

will still contain:

User prefers Python for AI development.
User is building an Agentic RAG project.

Keep the same:

user_id="sunny_123"

and ask:

You:
Which programming language do I prefer?

Flow:

Current Query
      ↓
SQLite
      ↓
Semantic Search
      ↓
"User prefers Python"
      ↓
LLM
      ↓
You prefer Python.

The distinction is now clear:

Memory	Storage	After restart
Recent messages	Python state	❌ Lost
Summary	Python state	❌ Lost
Full history	Python state	❌ Lost
Long-term user memory	SQLite	✅ Persists

Say this in class:

“For this demo, short-term conversation memory remains in application state, while long-term user memory is persisted in SQLite. Therefore, even if the application restarts, important user information can still be retrieved.”

What should we use in production?

For teaching/local:

SQLite ✅

Small standalone app:

SQLite ✅

Large multi-user production:

PostgreSQL / pgvector ✅

The official LangGraph guidance also recommends replacing the development InMemoryStore with a persistent database-backed Store such as AsyncPostgresStore in production.

If short-term/checkpoint state must also be persisted in production, LangGraph officially supports PostgresSaver / AsyncPostgresSaver for production workloads; the SQLite checkpointer is intended for lightweight demos and small projects.

So eventual full production architecture:

                PRODUCTION

Short-Term State
      ↓
PostgresSaver
      ↓
thread_id


Long-Term Memory
      ↓
PostgreSQL / pgvector
      ↓
user_id


Semantic Retrieval
      ↓
pgvector


Raw Conversation Archive
      ↓
PostgreSQL / S3

---
## 5. Persistence Versus Memory Scope

Why cross-thread scope and restart-safe persistence are separate concepts.

Yes, exactly.

store = InMemoryStore()

means:

Long-term memory scope ✅
Persistent storage ❌

This means memory can be reused across different threads within the same running process, but the data will be lost if the app/kernel/process restarts.

Simple distinction:

InMemoryStore
→ long-term by scope
→ not durable by storage

And:

Persistent Store
→ long-term by scope
→ durable across restart

Therefore, this is the best line to say in class:

“InMemoryStore can represent long-term memory across conversations, but it is not persistent. If the application restarts, the stored memories are lost.”

In production, you would use a persistent backend, for example:

PostgreSQL / Redis / MongoDB / DynamoDB / other persistent store

Short version:

Cross-thread memory ≠ automatically persistent memory

These are two different concepts.

---
## 6. Custom Long-Term Memory with LangGraph Store

Manual extraction, storage, retrieval, recent messages, summary, and full history.

Yes. We will retain the same memory + summary + full_history logic for short-term memory and add custom long-term memory using the LangGraph Store.

No LangMem. No Remember: keyword. We will extract, save, and retrieve long-term memory ourselves.

In [ ]:
# ============================================================
# INSTALL
# ============================================================
# pip install -U langgraph langchain-openai pydantic


# ============================================================
# IMPORTS
# ============================================================

import uuid

from dataclasses import dataclass
from typing import TypedDict, List, Dict

from pydantic import BaseModel, Field

from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage

from langgraph.graph import (
    StateGraph,
    START,
    END
)

from langgraph.store.memory import (
    InMemoryStore
)

from langgraph.runtime import (
    Runtime
)


# ============================================================
# MODEL
# ============================================================

model = ChatOpenAI(
    model="gpt-4.1-mini",
    temperature=0
)


# ============================================================
# CONFIG
# ============================================================

MAX_RECENT_MESSAGES = 6


# ============================================================
# USER CONTEXT
# ============================================================
#
# user_id tells us:
# WHO is the user?
#
# Long-term memory will be stored per user.
# ============================================================

@dataclass
class UserContext:

    user_id: str


# ============================================================
# LANGGRAPH STATE
# ============================================================
#
# This state manages CURRENT conversation memory.
# ============================================================

class MemoryState(TypedDict):

    user_input: str

    # Latest raw messages
    memory: List[Dict]

    # Compressed older conversation
    summary: str

    # Complete current conversation
    full_history: List[Dict]

    answer: str


# ============================================================
# LONG-TERM MEMORY STORE
# ============================================================
#
# This is different from conversation state.
#
# Store is user-level and can be reused
# across different conversations.
# ============================================================

store = InMemoryStore(
    index={
        "dims": 1536,
        "embed": "openai:text-embedding-3-small"
    }
)


# ============================================================
# LONG-TERM MEMORY EXTRACTION SCHEMA
# ============================================================

class ExtractedMemories(BaseModel):

    memories: List[str] = Field(
        default_factory=list,
        description="""
        Important long-term information
        learned about the user.
        """
    )


memory_extractor = model.with_structured_output(
    ExtractedMemories
)


# ============================================================
# CREATE / UPDATE SHORT-TERM SUMMARY
# ============================================================

def create_summary(
    old_messages,
    existing_summary
):

    conversation_text = "\n".join(

        f"{msg['role']}: {msg['content']}"

        for msg in old_messages
    )


    prompt = f"""
You maintain short-term conversation memory
for an AI assistant.

Existing summary:

{existing_summary or "No previous summary."}


Older conversation:

{conversation_text}


Create one concise updated summary.

Preserve:

- important conversation context
- important facts
- decisions
- projects
- preferences relevant to this conversation

Ignore:

- greetings
- unnecessary small talk
"""


    response = model.invoke(
        prompt
    )


    return response.content.strip()


# ============================================================
# EXTRACT LONG-TERM MEMORIES
# ============================================================
#
# Custom implementation.
# No LangMem.
# ============================================================

def extract_long_term_memories(
    user_message
):

    prompt = f"""
Analyze the following user message:

{user_message}


Extract only useful LONG-TERM information
about the user.

Examples of useful long-term memories:

- preferences
- personal facts
- goals
- ongoing projects
- skills
- recurring requirements
- important decisions

Do NOT store:

- greetings
- temporary questions
- general knowledge questions
- one-time requests
- unnecessary small talk


Example:

User:
I prefer Python for AI development.

Memory:
User prefers Python for AI development.


User:
What is RAG?

Memory:
Nothing.


User:
I am building an Agentic RAG project.

Memory:
User is building an Agentic RAG project.


If there is nothing useful to remember,
return an empty list.
"""


    result = memory_extractor.invoke(
        prompt
    )


    return result.memories


# ============================================================
# SAVE LONG-TERM MEMORY
# ============================================================

def save_long_term_memories(
    runtime,
    user_id,
    memories
):

    namespace = (
        "users",
        user_id,
        "memories"
    )


    # Get already stored memories

    existing_items = runtime.store.search(
        namespace
    )


    existing_memories = {

        item.value["content"]

        for item in existing_items
    }


    # Save only new memories

    for memory in memories:

        if memory not in existing_memories:

            memory_id = str(
                uuid.uuid4()
            )


            runtime.store.put(

                namespace,

                memory_id,

                {
                    "content": memory
                }
            )


# ============================================================
# SEARCH LONG-TERM MEMORY
# ============================================================

def search_long_term_memories(
    runtime,
    user_id,
    query,
    limit=5
):

    namespace = (
        "users",
        user_id,
        "memories"
    )


    results = runtime.store.search(

        namespace,

        query=query,

        limit=limit
    )


    return [

        item.value["content"]

        for item in results
    ]


# ============================================================
# NODE 1
# CHATBOT
# ============================================================

def chatbot(
    state: MemoryState,
    runtime: Runtime[UserContext]
):

    user_id = runtime.context.user_id


    # --------------------------------------------------------
    # COPY CURRENT SHORT-TERM MEMORY
    # --------------------------------------------------------

    memory = list(
        state["memory"]
    )


    full_history = list(
        state["full_history"]
    )


    # --------------------------------------------------------
    # ADD CURRENT USER MESSAGE
    # --------------------------------------------------------

    user_message = {

        "role": "user",

        "content": state["user_input"]
    }


    memory.append(
        user_message
    )


    full_history.append(
        user_message
    )


    # --------------------------------------------------------
    # SEARCH LONG-TERM MEMORY
    # --------------------------------------------------------

    long_term_memories = (
        search_long_term_memories(

            runtime=runtime,

            user_id=user_id,

            query=state["user_input"],

            limit=5
        )
    )


    if long_term_memories:

        long_term_text = "\n".join(

            f"- {memory_item}"

            for memory_item
            in long_term_memories
        )

    else:

        long_term_text = (
            "No relevant long-term memories."
        )


    # --------------------------------------------------------
    # SEND TO LLM:
    #
    # 1. Short-term summary
    # 2. Long-term memories
    # 3. Recent raw messages
    # --------------------------------------------------------

    system_message = SystemMessage(
        content=f"""
You are a helpful AI assistant.


SHORT-TERM CONVERSATION SUMMARY:

{state["summary"] or "No previous summary."}


RELEVANT LONG-TERM USER MEMORIES:

{long_term_text}


Use long-term memories only when relevant.
Use recent conversation messages to maintain
conversation continuity.
"""
    )


    response = model.invoke(

        [
            system_message,
            *memory
        ]
    )


    # --------------------------------------------------------
    # ADD ASSISTANT RESPONSE
    # --------------------------------------------------------

    assistant_message = {

        "role": "assistant",

        "content": response.content
    }


    memory.append(
        assistant_message
    )


    full_history.append(
        assistant_message
    )


    return {

        "memory": memory,

        "full_history": full_history,

        "answer": response.content
    }


# ============================================================
# NODE 2
# COMPRESS SHORT-TERM MEMORY
# ============================================================

def compress_memory(
    state: MemoryState
):

    memory = list(
        state["memory"]
    )


    # Nothing to compress yet

    if len(memory) <= MAX_RECENT_MESSAGES:

        return {}


    # --------------------------------------------------------
    # OLD MESSAGES
    # --------------------------------------------------------

    old_messages = memory[
        :-MAX_RECENT_MESSAGES
    ]


    # --------------------------------------------------------
    # KEEP LATEST 6 RAW MESSAGES
    # --------------------------------------------------------

    recent_messages = memory[
        -MAX_RECENT_MESSAGES:
    ]


    # --------------------------------------------------------
    # UPDATE SUMMARY
    # --------------------------------------------------------

    updated_summary = create_summary(

        old_messages,

        state["summary"]
    )


    return {

        "memory": recent_messages,

        "summary": updated_summary
    }


# ============================================================
# NODE 3
# UPDATE LONG-TERM MEMORY
# ============================================================

def update_long_term_memory(
    state: MemoryState,
    runtime: Runtime[UserContext]
):

    user_id = runtime.context.user_id


    # --------------------------------------------------------
    # EXTRACT IMPORTANT USER INFORMATION
    # --------------------------------------------------------

    memories = extract_long_term_memories(
        state["user_input"]
    )


    # --------------------------------------------------------
    # SAVE INTO LANGGRAPH STORE
    # --------------------------------------------------------

    save_long_term_memories(

        runtime=runtime,

        user_id=user_id,

        memories=memories
    )


    return {}


# ============================================================
# BUILD LANGGRAPH
# ============================================================

builder = StateGraph(

    MemoryState,

    context_schema=UserContext
)


builder.add_node(
    "chatbot",
    chatbot
)


builder.add_node(
    "compress_memory",
    compress_memory
)


builder.add_node(
    "update_long_term_memory",
    update_long_term_memory
)


# ============================================================
# GRAPH FLOW
# ============================================================

builder.add_edge(
    START,
    "chatbot"
)


builder.add_edge(
    "chatbot",
    "compress_memory"
)


builder.add_edge(
    "compress_memory",
    "update_long_term_memory"
)


builder.add_edge(
    "update_long_term_memory",
    END
)


# ============================================================
# COMPILE GRAPH
# ============================================================

graph = builder.compile(
    store=store
)


# ============================================================
# INITIAL SHORT-TERM STATE
# ============================================================

state = {

    "user_input": "",

    "memory": [],

    "summary": "",

    "full_history": [],

    "answer": ""
}


# ============================================================
# USER
# ============================================================

context = UserContext(
    user_id="sunny_123"
)


# ============================================================
# CHAT LOOP
# ============================================================

while True:

    user_input = input(
        "\nYou: "
    ).strip()


    if user_input.lower() in {
        "exit",
        "quit"
    }:

        print(
            "Chat ended."
        )

        break


    if not user_input:
        continue


    # Current input

    state["user_input"] = user_input


    # Run graph

    state = graph.invoke(

        state,

        context=context
    )


    # --------------------------------------------------------
    # OUTPUT
    # --------------------------------------------------------

    print(
        "\nAI:",
        state["answer"]
    )


    print(
        "\nRecent Messages:",
        len(state["memory"])
    )


    print(
        "Full History:",
        len(state["full_history"])
    )


    if state["summary"]:

        print(
            "\nCurrent Summary:\n",
            state["summary"]
        )

                         USER
                          │
                          ▼
                     User Input
                          │
                          ▼
                      CHATBOT
                          │
             ┌────────────┴────────────┐
             │                         │
             ▼                         ▼
       SHORT-TERM                  LONG-TERM
          MEMORY                     MEMORY
             │                         │
       summary + recent            Store.search()
          messages                     │
             │                         │
             └────────────┬────────────┘
                          ▼
                         LLM
                          │
                          ▼
                        Answer
                          │
             ┌────────────┴────────────┐
             │                         │
             ▼                         ▼
      Compress Memory         Extract User Facts
             │                         │
      Old → Summary                   │
      Latest 6 → Raw                  ▼
                               Store.put()

What exactly is short-term memory?
memory
→ latest 6 raw messages

summary
→ older conversation compressed

full_history
→ complete current conversation

For example after 16 messages:

full_history
→ M1 ... M16


summary
→ M1 ... M10 compressed


memory
→ M11 ... M16 raw

This is the memory of the current conversation/session.

What exactly is long-term memory?

Suppose the user says:

I prefer Python for AI development.

Custom extractor:

extract_long_term_memories(...)

will extract:

User prefers Python for AI development.

Then:

runtime.store.put(...)

saves it in the user's namespace:

users
 └── sunny_123
      └── memories
           └── User prefers Python for AI development.

Later query:

Which programming language do I prefer?

Then:

runtime.store.search(
    query=...
)

will retrieve semantically:

User prefers Python for AI development.

And it will be provided to the LLM.

The difference is now crystal clear:
Short-term	Long-term
Current conversation	Across conversations
memory	Store
summary	User facts/preferences
full_history	Semantic retrieval
Recent messages	Store.search()
Conversation context	User-level context

One caveat: this currently uses InMemoryStore(), so it demonstrates the cross-conversation concept, but long-term data will be lost when the process restarts. Use a persistent Store/backend in production.

Exact line for the class:

In this implementation, short-term memory is managed manually using recent messages, conversation summary, and full history, while long-term memory is managed separately in the LangGraph Store using custom extraction, storage, and semantic retrieval logic.

Yes, this code includes the concept of short-term memory, but it does not have proper long-term memory.

The fields are:

memory
summary
full_history

All of these are used only within the same running conversation/session.

So mapping:

memory
→ recent raw messages
→ short-term memory

summary
→ compressed context of older messages
→ still short-term/conversation memory

full_history
→ complete conversation record
→ it is history, but not automatically long-term user memory

Important point:

Creating a summary does not turn it into long-term memory.

Why? Because all of this exists inside the current Python state. If the app restarts:

memory = lost
summary = lost
full_history = lost

It is also not automatically reused in a new thread/new conversation.

Proper long-term memory requires separate persistent/user-level storage:

Short-Term Memory
→ recent messages + summary
→ current thread/session

Long-Term Memory
→ user facts/preferences/projects
→ across sessions/threads
→ DB / Store

Example:

User:
I prefer Python.

Current conversation:
→ can appear in recent memory
→ can appear in the summary

Long-term:
→ "User prefers Python"
→ saved in the Store/DB against the user_id
→ retrieved even in the next new chat

Therefore, your current code has:

✅ Recent memory
✅ Summary memory
✅ Full conversation history
❌ Cross-session long-term memory
❌ user_id based memory
❌ persistent Store/DB

If you want, I can modify this exact code to provide both short-term and long-term memory in a single LangGraph example, without LangMem, using simple custom logic.

---
## 7. Manual State-Based Memory in LangGraph

A custom-state implementation without `thread_id`, `user_id`, a checkpointer, or LangMem.

Yes, we can write exactly the same logic inside LangGraph with a custom state — memory, summary, and full_history all remain the same. This version uses no thread_id, user_id, checkpointer, or LangMem. We will manage memory ourselves in the state.

In [ ]:
# pip install -U langgraph langchain-openai

from typing import TypedDict, List, Dict

from langchain_openai import ChatOpenAI

from langgraph.graph import (
    StateGraph,
    START,
    END
)


# ============================================================
# MODEL
# ============================================================

model = ChatOpenAI(
    model="gpt-4.1-mini",
    temperature=0
)


# ============================================================
# CONFIG
# ============================================================

MAX_RECENT_MESSAGES = 6


# ============================================================
# LANGGRAPH STATE
# ============================================================

class MemoryState(TypedDict):

    user_input: str

    memory: List[Dict]

    summary: str

    full_history: List[Dict]

    answer: str


# ============================================================
# CREATE / UPDATE SUMMARY
# ============================================================

def create_summary(
    old_messages,
    existing_summary
):

    conversation_text = "\n".join(
        f"{msg['role']}: {msg['content']}"
        for msg in old_messages
    )


    prompt = f"""
You maintain conversation memory for an AI assistant.

Existing summary:

{existing_summary or "No previous summary."}


Older conversation:

{conversation_text}


Create one concise updated summary.

Preserve:
- important facts
- preferences
- projects
- decisions
- important conversation context

Ignore:
- greetings
- unnecessary small talk
"""


    response = model.invoke(
        prompt
    )


    return response.content.strip()


# ============================================================
# NODE 1
# CHATBOT
# ============================================================

def chatbot(
    state: MemoryState
):

    # ----------------------------------------
    # Copy existing memory
    # ----------------------------------------

    memory = list(
        state["memory"]
    )


    full_history = list(
        state["full_history"]
    )


    # ----------------------------------------
    # Add current user message
    # ----------------------------------------

    user_message = {
        "role": "user",
        "content": state["user_input"]
    }


    memory.append(
        user_message
    )


    full_history.append(
        user_message
    )


    # ----------------------------------------
    # Send summary + recent memory to LLM
    # ----------------------------------------

    messages = [

        {
            "role": "system",
            "content": f"""
Previous conversation summary:

{state["summary"] or "No previous summary."}
"""
        },

        *memory

    ]


    response = model.invoke(
        messages
    )


    # ----------------------------------------
    # Add assistant response
    # ----------------------------------------

    assistant_message = {
        "role": "assistant",
        "content": response.content
    }


    memory.append(
        assistant_message
    )


    full_history.append(
        assistant_message
    )


    return {

        "memory": memory,

        "full_history": full_history,

        "answer": response.content
    }


# ============================================================
# NODE 2
# MEMORY COMPRESSION
# ============================================================

def compress_memory(
    state: MemoryState
):

    memory = list(
        state["memory"]
    )


    summary = state["summary"]


    # ----------------------------------------
    # If memory is within limit,
    # do nothing
    # ----------------------------------------

    if len(memory) <= MAX_RECENT_MESSAGES:

        return {}


    # ----------------------------------------
    # Everything except latest 6
    # becomes old memory
    # ----------------------------------------

    old_messages = memory[
        :-MAX_RECENT_MESSAGES
    ]


    # ----------------------------------------
    # Keep latest 6 messages
    # ----------------------------------------

    recent_messages = memory[
        -MAX_RECENT_MESSAGES:
    ]


    # ----------------------------------------
    # Merge older messages
    # into existing summary
    # ----------------------------------------

    updated_summary = create_summary(

        old_messages,

        summary
    )


    return {

        "memory": recent_messages,

        "summary": updated_summary
    }


# ============================================================
# BUILD LANGGRAPH
# ============================================================

builder = StateGraph(
    MemoryState
)


builder.add_node(
    "chatbot",
    chatbot
)


builder.add_node(
    "compress_memory",
    compress_memory
)


# ============================================================
# GRAPH FLOW
# ============================================================

builder.add_edge(
    START,
    "chatbot"
)


builder.add_edge(
    "chatbot",
    "compress_memory"
)


builder.add_edge(
    "compress_memory",
    END
)


# ============================================================
# COMPILE
# ============================================================

graph = builder.compile()


# ============================================================
# INITIAL MEMORY STATE
# ============================================================

state = {

    "user_input": "",

    "memory": [],

    "summary": "",

    "full_history": [],

    "answer": ""
}


# ============================================================
# CHAT LOOP
# ============================================================

while True:

    user_input = input(
        "\nYou: "
    ).strip()


    if user_input.lower() in {
        "exit",
        "quit"
    }:

        print(
            "Chat ended."
        )

        break


    if not user_input:
        continue


    # ----------------------------------------
    # Give new user input
    # while preserving previous state
    # ----------------------------------------

    state["user_input"] = user_input


    # ----------------------------------------
    # Run LangGraph
    # ----------------------------------------

    state = graph.invoke(
        state
    )


    # ----------------------------------------
    # Output
    # ----------------------------------------

    print(
        "\nAI:",
        state["answer"]
    )


    print(
        "\nRecent Messages:",
        len(state["memory"])
    )


    print(
        "Full History:",
        len(state["full_history"])
    )


    if state["summary"]:

        print(
            "\nCurrent Summary:\n",
            state["summary"]
        )

This contains exactly the same logic:
state
│
├── memory
│     → latest 6 raw messages
│
├── summary
│     → compressed older messages
│
├── full_history
│     → complete conversation
│
└── user_input
      → current message

Graph flow:

User Input
    ↓
CHATBOT NODE
    ↓
Add user message
    ↓
Summary + Recent Memory
    ↓
LLM
    ↓
Add AI response
    ↓
COMPRESS MEMORY NODE
    ↓
memory > 6 ?
   /     \
 No      Yes
 ↓        ↓
Keep     Old Messages
         ↓
       Summary
         +
       Latest 6 raw messages

Suppose there are now 16 messages:

full_history
→ M1 ... M16

summary
→ M1 ... M10 compressed

memory
→ M11 ... M16 raw

This is exactly what we were doing earlier.

Important difference

This version:

Python variable `state`
→ maintains memory

Therefore:

Application restart
→ memory lost

Because we have not used:

checkpointer=...



Therefore, this is a custom LangGraph memory-management demo.

In production, the same logic can be placed:

Summary + Recent Messages

on top of a persistent database/checkpointer.

Say this in class:

LangGraph does not force us to use its built-in memory abstractions. We can manage memory ourselves inside the graph state using the same techniques such as full history, recent-message windows, and conversation summarization.

---
## 8. Pure LangGraph Custom Memory Manager

A checkpointer-and-Store implementation with custom memory extraction and retrieval.

Absolutely. LangMem is not compulsory. We can write pure LangGraph + custom memory-management code, just as we manually managed extraction, storage, and retrieval in a normal assistant/RAG application.

In this version:

LangGraph
→ graph + state + checkpointer + Store

Our Custom Code
→ decides what to remember
→ extract memories
→ store.put()
→ store.search()

We will not use create_memory_store_manager at all.

In [ ]:
# ============================================================
# INSTALL
# ============================================================
# pip install -U langgraph langchain-openai pydantic


# ============================================================
# IMPORTS
# ============================================================

import uuid

from dataclasses import dataclass
from typing import List

from pydantic import BaseModel, Field

from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage

from langgraph.graph import (
    StateGraph,
    MessagesState,
    START,
    END
)

from langgraph.checkpoint.memory import (
    InMemorySaver
)

from langgraph.store.memory import (
    InMemoryStore
)

from langgraph.runtime import Runtime


# ============================================================
# MODEL
# ============================================================

model = ChatOpenAI(
    model="gpt-4.1-mini",
    temperature=0
)


# ============================================================
# USER CONTEXT
# ============================================================
#
# user_id = WHO is the user?
#
# thread_id will separately identify
# WHICH conversation?
# ============================================================

@dataclass
class UserContext:

    user_id: str


# ============================================================
# SHORT-TERM MEMORY
# ============================================================
#
# Stores conversation state for each thread.
# ============================================================

checkpointer = InMemorySaver()


# ============================================================
# LONG-TERM MEMORY STORE
# ============================================================
#
# Vector-search-enabled LangGraph Store.
#
# Stores useful information across threads.
# ============================================================

store = InMemoryStore(
    index={
        "dims": 1536,
        "embed": "openai:text-embedding-3-small"
    }
)


# ============================================================
# MEMORY EXTRACTION SCHEMA
# ============================================================

class ExtractedMemories(BaseModel):

    memories: List[str] = Field(
        default_factory=list,
        description="""
        Important long-term information
        learned about the user.
        """
    )


# ============================================================
# STRUCTURED MEMORY EXTRACTION MODEL
# ============================================================

memory_extractor = model.with_structured_output(
    ExtractedMemories
)


# ============================================================
# CUSTOM MEMORY EXTRACTION
# ============================================================
#
# THIS replaces LangMem.
#
# We decide ourselves what should
# become long-term memory.
# ============================================================

def extract_memories(
    user_message: str
):

    prompt = f"""
Analyze the following user message.

User message:
{user_message}

Extract only useful long-term information
about the user.

Useful memories may include:

- preferences
- personal facts
- goals
- projects
- skills
- recurring requirements
- important decisions

Do NOT store:

- greetings
- temporary questions
- general knowledge questions
- one-time requests
- unnecessary small talk

Examples:

User:
I prefer Python for AI projects.

Memory:
User prefers Python for AI projects.


User:
What is LangGraph?

Memory:
Nothing.


User:
I am building an Agentic RAG system.

Memory:
User is building an Agentic RAG system.


If there is nothing useful to remember,
return an empty list.
"""

    result = memory_extractor.invoke(
        prompt
    )

    return result.memories


# ============================================================
# CUSTOM LONG-TERM MEMORY SAVE
# ============================================================

def save_memories(
    runtime,
    user_id,
    memories
):

    namespace = (
        "users",
        user_id,
        "memories"
    )


    # ----------------------------------------
    # Read existing memories
    # ----------------------------------------

    existing_items = runtime.store.search(
        namespace
    )


    existing_memories = {
        item.value["content"]
        for item in existing_items
    }


    # ----------------------------------------
    # Save only new memories
    # ----------------------------------------

    for memory in memories:

        if memory not in existing_memories:

            memory_id = str(
                uuid.uuid4()
            )

            runtime.store.put(
                namespace,
                memory_id,
                {
                    "content": memory
                }
            )


# ============================================================
# CUSTOM LONG-TERM MEMORY SEARCH
# ============================================================

def search_memories(
    runtime,
    user_id,
    query,
    limit=5
):

    namespace = (
        "users",
        user_id,
        "memories"
    )


    results = runtime.store.search(
        namespace,
        query=query,
        limit=limit
    )


    return [
        item.value["content"]
        for item in results
    ]


# ============================================================
# CHATBOT NODE
# ============================================================

def chatbot(
    state: MessagesState,
    runtime: Runtime[UserContext]
):

    # ----------------------------------------
    # WHO is the user?
    # ----------------------------------------

    user_id = runtime.context.user_id


    # ----------------------------------------
    # Latest user query
    # ----------------------------------------

    user_query = (
        state["messages"][-1].content
    )


    # ----------------------------------------
    # Retrieve relevant long-term memories
    # ----------------------------------------

    memories = search_memories(
        runtime=runtime,
        user_id=user_id,
        query=user_query,
        limit=5
    )


    # ----------------------------------------
    # Format memories
    # ----------------------------------------

    if memories:

        memory_text = "\n".join(
            f"- {memory}"
            for memory in memories
        )

    else:

        memory_text = (
            "No relevant long-term memories."
        )


    # ----------------------------------------
    # Build system prompt
    # ----------------------------------------

    system_message = SystemMessage(
        content=f"""
You are a helpful AI assistant.

Relevant long-term memories about the user:

{memory_text}

Use these memories only when relevant.

The remaining messages contain the
conversation history of the current thread.
"""
    )


    # ----------------------------------------
    # LLM CALL
    #
    # state["messages"] already contains
    # current thread conversation history.
    # ----------------------------------------

    response = model.invoke(
        [
            system_message,
            *state["messages"]
        ]
    )


    return {
        "messages": [
            response
        ]
    }


# ============================================================
# CUSTOM MEMORY MANAGEMENT NODE
# ============================================================

def memory_manager(
    state: MessagesState,
    runtime: Runtime[UserContext]
):

    user_id = runtime.context.user_id


    # ----------------------------------------
    # Find latest user message
    # ----------------------------------------

    latest_user_message = None


    for message in reversed(
        state["messages"]
    ):

        if message.type == "human":

            latest_user_message = (
                message.content
            )

            break


    if not latest_user_message:

        return {}


    # ----------------------------------------
    # STEP 1:
    # Extract useful memories
    # ----------------------------------------

    memories = extract_memories(
        latest_user_message
    )


    # ----------------------------------------
    # STEP 2:
    # Save memories into LangGraph Store
    # ----------------------------------------

    save_memories(
        runtime=runtime,
        user_id=user_id,
        memories=memories
    )


    return {}


# ============================================================
# BUILD GRAPH
# ============================================================

builder = StateGraph(
    MessagesState,
    context_schema=UserContext
)


builder.add_node(
    "chatbot",
    chatbot
)


builder.add_node(
    "memory_manager",
    memory_manager
)


# ============================================================
# GRAPH FLOW
# ============================================================

builder.add_edge(
    START,
    "chatbot"
)


builder.add_edge(
    "chatbot",
    "memory_manager"
)


builder.add_edge(
    "memory_manager",
    END
)


# ============================================================
# COMPILE GRAPH
# ============================================================

graph = builder.compile(

    # Short-term memory
    checkpointer=checkpointer,

    # Long-term memory
    store=store
)


# ============================================================
# CHAT FUNCTION
# ============================================================

def chat(
    user_id,
    thread_id,
    message
):

    # thread_id identifies conversation

    config = {
        "configurable": {
            "thread_id": thread_id
        }
    }


    # user_id identifies user

    context = UserContext(
        user_id=user_id
    )


    result = graph.invoke(

        {
            "messages": [
                {
                    "role": "user",
                    "content": message
                }
            ]
        },

        config=config,

        context=context
    )


    return (
        result["messages"][-1].content
    )


# ============================================================
# TEST 1
# THREAD 1
# ============================================================

print(
    chat(
        user_id="sunny_123",
        thread_id="thread_1",
        message=(
            "My favorite programming "
            "language is Python."
        )
    )
)


print(
    chat(
        user_id="sunny_123",
        thread_id="thread_1",
        message=(
            "I am currently building "
            "an Agentic RAG application."
        )
    )
)


# ============================================================
# TEST 2
# SAME THREAD
#
# Checkpointer provides conversation memory.
# ============================================================

print(
    chat(
        user_id="sunny_123",
        thread_id="thread_1",
        message=(
            "What project am I building?"
        )
    )
)


# ============================================================
# TEST 3
# NEW THREAD
# SAME USER
#
# Checkpointer history is different,
# but Store can retrieve long-term memories.
# ============================================================

print(
    chat(
        user_id="sunny_123",
        thread_id="thread_2",
        message=(
            "Which programming language "
            "do I prefer?"
        )
    )
)


# ============================================================
# TEST 4
# DIFFERENT USER
# ============================================================

print(
    chat(
        user_id="user_456",
        thread_id="thread_1",
        message=(
            "Which programming language "
            "do I prefer?"
        )
    )
)

What exactly is this code doing?

In the code that uses LangMem:

Conversation
    ↓
create_memory_store_manager()
    ↓
Extract memory
    ↓
Update Store

Now we have removed LangMem:

Conversation
     ↓
extract_memories()
     ↓
Our LLM-based extraction
     ↓
save_memories()
     ↓
runtime.store.put()

We also handle retrieval ourselves:

Current User Query
       ↓
search_memories()
       ↓
runtime.store.search()
       ↓
Top 5 relevant memories
       ↓
System Prompt
       ↓
LLM

Therefore, the complete architecture is:

                        USER
                         │
                         ▼
                 Current Message
                         │
          ┌──────────────┴──────────────┐
          │                             │
          ▼                             ▼
     Checkpointer                    Store
          │                             │
     thread_id                       user_id
          │                             │
          ▼                             ▼
 Current Conversation          Long-Term Memories
          │                             │
          └──────────────┬──────────────┘
                         ▼
                     Chatbot
                         │
                         ▼
                        LLM
                         │
                         ▼
                      Answer
                         │
                         ▼
                Memory Manager Node
                         │
                         ▼
                extract_memories()
                         │
                         ▼
                  save_memories()
                         │
                         ▼
                    Store.put()
Example

User says:

My favorite programming language is Python.

Our custom function:

extract_memories(...)

LLM may return:

[
    "User's favorite programming language is Python."
]

Then:

save_memories(...)

does:

runtime.store.put(...)

So Store:

users
 └── sunny_123
      └── memories
           └── User's favorite programming language is Python.

New thread:

thread_2

User:
Which programming language do I prefer?

Then:

search_memories(
    query="Which programming language do I prefer?"
)

semantic search returns:

User's favorite programming language is Python.

and LLM gets:

Relevant long-term memories:

- User's favorite programming language is Python.

Current conversation:
Human: Which programming language do I prefer?

Then response:

You prefer Python.
LangMem vs Custom Code
LangMem	Custom implementation
create_memory_store_manager()	extract_memories() + save_memories()
Automatic extraction	We write extraction prompt
Automatic Store management	We call store.put()
Built-in update logic	We implement update/dedup logic
Less code	More code
Abstraction	Full control

Therefore, you can say this in class:

“LangMem is optional. LangGraph provides the Store abstraction, but we can build our own memory-management layer on top of it. We can use an LLM to extract important information, Store.put() to save it, and Store.search() to retrieve relevant memories.”

---
## 9. LangGraph Versus LangMem

A final responsibility breakdown and implementation comparison.

Yes, the practical I provided uses both LangGraph and LangMem.

Breakdown:

LangGraph
→ graph/state orchestration
→ MessagesState
→ StateGraph
→ Checkpointer
→ Store

LangMem
→ automatic long-term memory extraction/update
→ create_memory_store_manager

This means:

from langmem import create_memory_store_manager

this part belongs to LangMem.

And:

from langgraph.graph import StateGraph, MessagesState
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.store.memory import InMemoryStore

this belongs to LangGraph.

Therefore, the architecture of the practical is:

User Message
    ↓
LangGraph State
    ↓
Chatbot Node
    ↓
LLM Response
    ↓
LangMem Memory Manager
    ↓
Useful long-term memory extract/update
    ↓
LangGraph Store

Say this exact line in class:

“This practical uses LangGraph for state and workflow management, and LangMem for automatic long-term memory extraction and management.”

If you want, I can also provide a version without LangMem that uses the pure LangGraph Store with manual memory extraction.